In [3]:
import pandas as pd

# 读取 myctrls_quarterly.dta
myctrls_quarterly = pd.read_stata(r"../公司财务/数据/上市公司财务数据/上市公司财务数据.dta")
# 随机挑100行完整输出，不要有...
# pd.set_option("display.max_columns", None)
# myctrls_quarterly.sample(100).to_csv("myctrls_quarterly.csv", index=False)
myctrls_quarterly

,stkcd,ShortName,Accper,Indcd,Indnme,Indcd1,Indnme1,rliquid,rcash,rar,...,IncomeTax,ownership,fyear,tempdt,quarter,tq,soe,研发费用,短期借款,长期借款
0,1,深发展A,1990-12-31,J66,货币金融服务,J66,货币金融服务,0.275487,NaN,0.000000,...,NaN,私营企业,1990,12,4,199004,0.0,NaN,0.000000e+00,NaN
1,1,深发展A,1991-12-31,J66,货币金融服务,J66,货币金融服务,0.007601,NaN,0.000000,...,NaN,私营企业,1991,12,4,199104,0.0,NaN,0.000000e+00,NaN
2,1,深发展A,1992-12-31,J66,货币金融服务,J66,货币金融服务,0.008402,NaN,0.000000,...,NaN,私营企业,1992,12,4,199204,0.0,NaN,0.000000e+00,NaN
3,1,深发展A,1993-12-31,J66,货币金融服务,J66,货币金融服务,NaN,NaN,0.000000,...,47150639.79,私营企业,1993,12,4,199304,0.0,NaN,0.000000e+00,NaN
4,1,深发展A,1994-06-30,J66,货币金融服务,J66,货币金融服务,NaN,NaN,0.062131,...,NaN,私营企业,1994,6,2,199402,0.0,NaN,0.000000e+00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284893,920819,颖泰生物,2024-03-31,,,C26,化学原料和化学制品制造业,0.391581,0.068541,0.095623,...,5983243.93,私营企业,2024,3,1,202401,0.0,3.848585e+07,3.700420e+09,8.569099e+08
284894,920819,颖泰生物,2024-06-30,,,C26,化学原料和化学制品制造业,0.391227,0.069472,0.103848,...,6856261.20,私营企业,2024,6,2,202402,0.0,8.506676e+07,3.638115e+09,1.025789e+09
284895,920819,颖泰生物,2024-09-30,,,C26,化学原料和化学制品制造业,0.382644,0.044653,0.120556,...,-4263159.72,私营企业,2024,9,3,202403,0.0,1.295415e+08,2.985886e+09,1.155837e+09
284896,920819,颖泰生物,2024-12-31,,,C26,化学原料和化学制品制造业,0.382142,0.044419,0.119890,...,-42522270.55,私营企业,2024,12,4,202404,0.0,1.734287e+08,2.612916e+09,8.491461e+08


In [5]:
import numpy as np
import pandas as pd

from linearmodels.panel import PanelOLS

INNOV_PATH = r"../公司财务/firm_year_innovation.parquet"
FIN_PATH   = r"../公司财务/数据/上市公司财务数据/上市公司财务数据.dta"

YEAR_MIN, YEAR_MAX = 2000, 2023


1) 读取创新 firm-year 数据（你算出来的）

In [8]:
innov = pd.read_parquet(INNOV_PATH)

# 统一列名/类型
innov = innov.rename(columns={
    "Stkid": "stkcd",
    "ShortName": "shortname",
})

innov["stkcd"] = innov["stkcd"].astype(int).astype(str).str.zfill(6)
innov["year"]  = innov["year"].astype(int)

# 可选：只保留你要回归的列
innov = innov[["stkcd", "year", "Innovation_raw", "Innovation_z", "PatentCount", "Method", "shortname"]]

print(innov.shape)
innov.head()


(29309, 7)


,stkcd,year,Innovation_raw,Innovation_z,PatentCount,Method,shortname
0,300057,2010,3.157372,-0.077261,2,Top10Mean,万顺新材
1,600019,2009,5.902089,2.562347,235,Top10Mean,宝钢股份
2,600884,2013,3.024427,0.641510,13,Top10Mean,杉杉股份
3,603568,2018,0.907066,-0.752391,1,Top10Mean,伟明环保
4,600831,2017,1.960893,0.927437,36,Top10Mean,广电网络


2) 读取财务数据（季度/年报混合），只保留年报（12-31）

In [9]:
fin = pd.read_stata(FIN_PATH)

# 关键列检查
need_cols = ["stkcd", "Accper", "roa", "roe", "tq", "asset", "liability", "finlev", "gassets", "soe"]
missing = [c for c in need_cols if c not in fin.columns]
print("Missing cols:", missing)
print(fin.shape)
fin.head()


Missing cols: []
(284898, 41)


,stkcd,ShortName,Accper,Indcd,Indnme,Indcd1,Indnme1,rliquid,rcash,rar,...,IncomeTax,ownership,fyear,tempdt,quarter,tq,soe,研发费用,短期借款,长期借款
0,1,深发展A,1990-12-31,J66,货币金融服务,J66,货币金融服务,0.275487,NaN,0.000000,...,NaN,私营企业,1990,12,4,199004,0.0,NaN,0.0,NaN
1,1,深发展A,1991-12-31,J66,货币金融服务,J66,货币金融服务,0.007601,NaN,0.000000,...,NaN,私营企业,1991,12,4,199104,0.0,NaN,0.0,NaN
2,1,深发展A,1992-12-31,J66,货币金融服务,J66,货币金融服务,0.008402,NaN,0.000000,...,NaN,私营企业,1992,12,4,199204,0.0,NaN,0.0,NaN
3,1,深发展A,1993-12-31,J66,货币金融服务,J66,货币金融服务,NaN,NaN,0.000000,...,47150639.79,私营企业,1993,12,4,199304,0.0,NaN,0.0,NaN
4,1,深发展A,1994-06-30,J66,货币金融服务,J66,货币金融服务,NaN,NaN,0.062131,...,NaN,私营企业,1994,6,2,199402,0.0,NaN,0.0,NaN


In [12]:
# 处理日期与年份
fin["Accper"] = pd.to_datetime(fin["Accper"], errors="coerce")
fin["year"]   = fin["Accper"].dt.year # pyright: ignore[reportAttributeAccessIssue]
fin["month"]  = fin["Accper"].dt.month # pyright: ignore[reportAttributeAccessIssue]
fin["day"]    = fin["Accper"].dt.day # pyright: ignore[reportAttributeAccessIssue]

# 只保留年报（12-31）
fin_annual = fin[(fin["month"] == 12) & (fin["day"] == 31)].copy()

# 限制样本年份（与你的创新表一致）
fin_annual = fin_annual[(fin_annual["year"] >= YEAR_MIN) & (fin_annual["year"] <= YEAR_MAX)].copy()

# 统一股票代码格式
fin_annual["stkcd"] = fin_annual["stkcd"].astype(int).astype(str).str.zfill(6)

# 同一公司同一年万一有重复（极少）：取最后一条
fin_annual = fin_annual.sort_values(["stkcd", "Accper"]).drop_duplicates(["stkcd", "year"], keep="last")

print(fin_annual.shape)
fin_annual.head()


(66305, 44)


,stkcd,ShortName,Accper,Indcd,Indnme,Indcd1,Indnme1,rliquid,rcash,rar,...,tempdt,quarter,tq,soe,研发费用,短期借款,长期借款,year,month,day
17,000001,深发展A,2000-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.113418,-0.005890,...,12,4,200004,0.0,NaN,NaN,NaN,2000,12,31
19,000001,深发展A,2001-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.182733,0.000093,...,12,4,200104,0.0,NaN,NaN,NaN,2001,12,31
23,000001,深发展A,2002-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.140819,-0.004590,...,12,4,200204,0.0,NaN,NaN,NaN,2002,12,31
27,000001,深发展A,2003-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.099071,-0.004320,...,12,4,200304,0.0,NaN,NaN,NaN,2003,12,31
31,000001,深发展A,2004-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.076506,-0.004280,...,12,4,200404,0.0,NaN,NaN,NaN,2004,12,31


3) 构造控制变量（尽量用你表里已有的）

推荐控制变量（你之前确认的口径）：

ln_asset = ln(asset)

lev = liability/asset（或直接用 finlev）

gassets

（可选）soe（国企 dummy）

In [13]:
df = fin_annual.merge(innov, on=["stkcd", "year"], how="inner")  # 只保留“有创新”的 firm-year
print("Merged:", df.shape)
df.head()


Merged: (28801, 49)


,stkcd,ShortName,Accper,Indcd,Indnme,Indcd1,Indnme1,rliquid,rcash,rar,...,短期借款,长期借款,year,month,day,Innovation_raw,Innovation_z,PatentCount,Method,shortname
0,000001,平安银行,2013-12-31,J66,货币金融服务,J66,货币金融服务,0.0,0.095734,0.101343,...,0.0,0.0,2013,12,31,2.626224,0.059144,1,Top10Mean,平安银行
1,000001,平安银行,2017-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.042181,0.130895,...,NaN,NaN,2017,12,31,2.320276,1.650289,4,Top10Mean,平安银行
2,000001,平安银行,2018-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.047330,0.000000,...,NaN,NaN,2018,12,31,1.638700,1.068504,1,Top10Mean,平安银行
3,000001,平安银行,2019-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.045457,0.000000,...,NaN,NaN,2019,12,31,1.314839,1.550971,54,Top10Mean,平安银行
4,000001,平安银行,2020-12-31,J66,货币金融服务,J66,货币金融服务,NaN,0.062426,0.000000,...,NaN,NaN,2020,12,31,0.666968,1.430051,112,Top10Mean,平安银行


In [14]:
# 基础清洗：资产为正
df = df[df["asset"].notna() & (df["asset"] > 0)].copy()

# 规模
df["ln_asset"] = np.log(df["asset"])

# 杠杆（两种都留着：lev_ratio / finlev，回归时选其一）
df["lev_ratio"] = df["liability"] / df["asset"]

# 可选：SOE 转为 0/1（如果原来就是 0/1 可跳过）
if df["soe"].dtype != np.int64 and df["soe"].dtype != np.int32:
    # 兜底：如果 soe 是字符串/类别，尝试转
    df["soe"] = pd.to_numeric(df["soe"], errors="coerce")
df["soe"] = df["soe"].fillna(0).astype(int)

# 丢掉回归用变量缺失的行（你也可以按模型逐个 dropna）
print(df[["roa","Innovation_z","ln_asset","lev_ratio","gassets"]].isna().mean())


roa             0.000000
Innovation_z    0.000000
ln_asset        0.000000
lev_ratio       0.000000
gassets         0.000104
dtype: float64


4) 设置面板索引 + 生成滞后项（lag1 / lag2）

In [15]:
df = df.sort_values(["stkcd", "year"]).copy()
df = df.set_index(["stkcd", "year"])

# 生成滞后创新（按公司）
df["Innovation_z_lag1"] = df.groupby(level=0)["Innovation_z"].shift(1)
df["Innovation_z_lag2"] = df.groupby(level=0)["Innovation_z"].shift(2)

# 可选：专利数量也可以 lag
df["PatentCount_lag1"]  = df.groupby(level=0)["PatentCount"].shift(1)

df.head()


ShortName     Accper Indcd  Indnme Indcd1 Indnme1  rliquid  \
stkcd  year                                                              
000001 2013      平安银行 2013-12-31   J66  货币金融服务    J66  货币金融服务      0.0   
       2017      平安银行 2017-12-31   J66  货币金融服务    J66  货币金融服务      NaN   
       2018      平安银行 2018-12-31   J66  货币金融服务    J66  货币金融服务      NaN   
       2019      平安银行 2019-12-31   J66  货币金融服务    J66  货币金融服务      NaN   
       2020      平安银行 2020-12-31   J66  货币金融服务    J66  货币金融服务      NaN   

                rcash       rar    rfixed  ...  Innovation_raw  Innovation_z  \
stkcd  year                                ...                                 
000001 2013  0.095734  0.101343  0.001953  ...        2.626224      0.059144   
       2017  0.042181  0.130895  0.002474  ...        2.320276      1.650289   
       2018  0.047330  0.000000  0.003188  ...        1.638700      1.068504   
       2019  0.045457  0.000000  0.002816  ...        1.314839      1.550971   
       2020  0.062426  0.000000  0.002438  ...        0.666968      1.430051   

             PatentCount     Method  shortname   ln_asset  lev_ratio  \
stkcd  year                                                            
000001 2013            1  Top10Mean       平安银行  28.268519   0.940752   
       2017            4  Top10Mean       平安银行  28.809206   0.931644   
       2018            1  Top10Mean       平安银行  28.860250   0.929783   
       2019           54  Top10Mean       平安银行  29.001966   0.920544   
       2020          112  Top10Mean       平安银行  29.128077   0.918512   

             Innovation_z_lag1  Innovation_z_lag2  PatentCount_lag1  
stkcd  year                                                          
000001 2013                NaN                NaN               NaN  
       2017           0.059144                NaN               1.0  
       2018           1.650289           0.059144               4.0  
       2019           1.068504           1.650289               1.0  
       2020           1.550971           1.068504              54.0  

[5 rows x 52 columns]

5) 跑回归（reghdfe 同款：公司FE + 年FE + 公司聚类SE）

5.1 Baseline：ROA ~ Innovation_z + FE

In [16]:
reg_df = df[["roa", "Innovation_z"]].dropna()

m1 = PanelOLS.from_formula(
    "roa ~ 1 + Innovation_z + EntityEffects + TimeEffects",
    data=reg_df
)
r1 = m1.fit(cov_type="clustered", cluster_entity=True)
print(r1.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                     3.553e-05
Estimator:                   PanelOLS   R-squared (Between):             -0.0002
No. Observations:               28801   R-squared (Within):            2.697e-05
Date:                  周日, 2月 08 2026   R-squared (Overall):          -4.675e-09
Time:                        12:42:45   Log-likelihood                 1.317e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.8613
Entities:                        4535   P-value                           0.3534
Avg Obs:                       6.3508   Distribution:                 F(1,24242)
Min Obs:                       1.0000                                           
Max Obs:                       24.000   F-statistic (robust):             2.3802
                            

5.2 主规格：加控制变量（建议别堆太多）

B) 用 finlev（如果你更信这个字段）

In [20]:
reg_df = df[["roa", "Innovation_z", "ln_asset", "finlev", "gassets"]].dropna()

m2 = PanelOLS.from_formula(
    "roa ~ 1 + Innovation_z + ln_asset + finlev + gassets + EntityEffects + TimeEffects",
    data=reg_df
)
r2 = m2.fit(cov_type="clustered", cluster_entity=True)
print(r2.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                        0.0076
Estimator:                   PanelOLS   R-squared (Between):             -0.0001
No. Observations:               25583   R-squared (Within):               0.0030
Date:                  周日, 2月 08 2026   R-squared (Overall):           3.127e-05
Time:                        12:44:36   Log-likelihood                 1.502e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      40.590
Entities:                        4388   P-value                           0.0000
Avg Obs:                       5.8302   Distribution:                 F(4,21168)
Min Obs:                       1.0000                                           
Max Obs:                       24.000   F-statistic (robust):             3.5482
                            

5.3 滞后创新：ROA ~ L1(Innovation_z) + controls + FE

In [18]:
reg_df = df[["roa", "Innovation_z_lag1", "ln_asset", "lev_ratio", "gassets"]].dropna()

m3 = PanelOLS.from_formula(
    "roa ~ 1 + Innovation_z_lag1 + ln_asset + lev_ratio + gassets + EntityEffects + TimeEffects",
    data=reg_df
)
r3 = m3.fit(cov_type="clustered", cluster_entity=True)
print(r3.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                        0.1046
Estimator:                   PanelOLS   R-squared (Between):              0.0207
No. Observations:               24265   R-squared (Within):               0.1070
Date:                  周日, 2月 08 2026   R-squared (Overall):              0.0727
Time:                        12:43:30   Log-likelihood                 1.084e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      593.32
Entities:                        3917   P-value                           0.0000
Avg Obs:                       6.1948   Distribution:                 F(4,20322)
Min Obs:                       1.0000                                           
Max Obs:                       23.000   F-statistic (robust):             1438.3
                            

5.4 稳健性：换因变量（ROE / TQ）

ROE：

In [19]:
reg_df = df[["roe", "Innovation_z", "ln_asset", "lev_ratio", "gassets"]].dropna()

m4 = PanelOLS.from_formula(
    "roe ~ 1 + Innovation_z + ln_asset + lev_ratio + gassets + EntityEffects + TimeEffects",
    data=reg_df
)
r4 = m4.fit(cov_type="clustered", cluster_entity=True)
print(r4.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    roe   R-squared:                        0.0043
Estimator:                   PanelOLS   R-squared (Between):             -0.3205
No. Observations:               28706   R-squared (Within):               0.0011
Date:                  周日, 2月 08 2026   R-squared (Overall):             -0.0458
Time:                        12:43:57   Log-likelihood                -8.174e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      26.006
Entities:                        4531   P-value                           0.0000
Avg Obs:                       6.3355   Distribution:                 F(4,24148)
Min Obs:                       1.0000                                           
Max Obs:                       24.000   F-statistic (robust):             2.5405
                            

6)（可选）把回归结果整理成一个小表（系数/SE/样本量）

In [21]:
def pick(res, var="Innovation_z"):
    out = {
        "coef": res.params.get(var, np.nan),
        "se": res.std_errors.get(var, np.nan),
        "t": res.tstats.get(var, np.nan),
        "p": res.pvalues.get(var, np.nan),
        "nobs": int(res.nobs),
        "rsq_within": float(res.rsquared_within),
    }
    return out

summary_table = pd.DataFrame({
    "ROA Baseline": pick(r1, "Innovation_z"),
    "ROA + Ctrls": pick(r2, "Innovation_z"),
    "ROA Lag1":    pick(r3, "Innovation_z_lag1"),
    "ROE + Ctrls": pick(r4, "Innovation_z"),
}).T

summary_table


,coef,se,t,p,nobs,rsq_within
ROA Baseline,0.001119,0.000725,1.542788,0.122895,28801.0,0.000027
ROA + Ctrls,0.001594,0.000566,2.815849,0.004869,25583.0,0.003003
ROA Lag1,0.002284,0.001553,1.470726,0.141381,24265.0,0.106995
ROE + Ctrls,0.014803,0.021127,0.700687,0.483505,28706.0,0.001085


比较研发费用占比

In [22]:
df["rd_intensity"] = df["研发费用"] / df["asset"]

# 可选：处理极端值
df["rd_intensity"] = df["rd_intensity"].clip(upper=0.5)

In [23]:
from linearmodels.panel import PanelOLS

reg_df = df[
    ["roa", "Innovation_z", "rd_intensity",
     "ln_asset", "lev_ratio", "gassets"]
].dropna()

m_rd = PanelOLS.from_formula(
    "roa ~ 1 + Innovation_z + rd_intensity "
    "+ ln_asset + lev_ratio + gassets "
    "+ EntityEffects + TimeEffects",
    data=reg_df
)

r_rd = m_rd.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(r_rd.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    roa   R-squared:                        0.1639
Estimator:                   PanelOLS   R-squared (Between):              0.2728
No. Observations:               14782   R-squared (Within):               0.1701
Date:                  周日, 2月 08 2026   R-squared (Overall):              0.2000
Time:                        13:10:05   Log-likelihood                 2.097e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      415.12
Entities:                        4181   P-value                           0.0000
Avg Obs:                       3.5355   Distribution:                 F(5,10589)
Min Obs:                       1.0000                                           
Max Obs:                       7.0000   F-statistic (robust):             12.679
                            